# Photo → Motion: CogVideoX-5B-I2V

Бесплатный экспериментальный режим для телефона через Google Colab. Нажмите Runtime → Change runtime type → T4 GPU, затем запускайте ячейки по очереди. Бесплатный GPU Colab предоставляется с ограничениями и может быть недоступен.

In [ ]:
!pip -q install --upgrade diffusers transformers accelerate imageio-ffmpeg
import torch
print('CUDA:', torch.cuda.is_available(), '| GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'не найден')

In [ ]:
from google.colab import files
uploaded = files.upload()
image_path = next(iter(uploaded))
print('Загружено:', image_path)

In [ ]:
prompt = input('Опишите движение на английском: ')
if not prompt.strip():
    prompt = 'Slow cinematic camera push-in, gentle natural movement, realistic lighting.'
print('Prompt:', prompt)

In [ ]:
import torch
from diffusers import CogVideoXImageToVideoPipeline
from diffusers.utils import load_image, export_to_video
from IPython.display import Video, display

model_id = 'THUDM/CogVideoX-5b-I2V'
pipe = CogVideoXImageToVideoPipeline.from_pretrained(model_id, torch_dtype=torch.bfloat16)
pipe.enable_sequential_cpu_offload()
pipe.vae.enable_tiling()
pipe.vae.enable_slicing()
image = load_image(image_path)
video = pipe(prompt=prompt, image=image, num_videos_per_prompt=1, num_inference_steps=50, num_frames=49, guidance_scale=6, generator=torch.Generator(device='cuda').manual_seed(42)).frames[0]
export_to_video(video, 'photo-to-motion.mp4', fps=8)
display(Video('photo-to-motion.mp4', embed=True))

In [ ]:
from google.colab import files
files.download('photo-to-motion.mp4')